# FASSD Multi-Label Segmentation Ensemble Study

This benchmark notebook presents the architectural ensembling study for the FASSD dataset, combining checkpoints from Run 1 and Round 2 across complementary decoder backbones (`U-Net`, `U-Net++`, `FPN`) and encoders (`efficientnet-b4`, `resnet34`).

### Protocol & Integrity Rules
1. **Model Checkpoints**:
   - `fassd_gray_seed42`: Run 1 U-Net (`efficientnet-b4`, seed 42)
   - `run2_unet_effb4`: Round 2 U-Net (`efficientnet-b4`, seed 42)
   - `run2_unetplusplus_effb4`: Round 2 U-Net++ (`efficientnet-b4`, seed 42)
   - `run2_unet_resnet34`: Round 2 U-Net (`resnet34`, seed 42)
   - `run2_fpn_effb4`: Round 2 FPN (`efficientnet-b4`, seed 42)
   - *RGB Exclusion*: Run 1 RGB is excluded to ensure strictly uniform single-channel grayscale preprocessing across all ensemble members.
2. **Probability-Average Ensembling**: Predictions are generated at native $768 \times 1024$ resolution and ensembled by arithmetic probability averaging.
3. **Validation Selection**: Candidates (`All 5 Gray Models`, `Top-3 by VAL Dice`, `Top-2 by VAL Dice`) are evaluated strictly on the validation set (254 images) with macro Dice only.
4. **Single Test Evaluation**: The winning ensemble is evaluated on the test set (271 images) **once**, alongside the best single validation model, computing Dice, IoU, and 95% Hausdorff Distance.
5. **Statistical Significance**: Patient-level paired bootstrap (5,000 resamples, seed 42) for $\Delta \text{Dice}$ and $\Delta \text{HD95}$ with 95% confidence intervals.


In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

RESULTS_JSON = Path("../../docs/results/fassd/ensemble/fassd_ensemble_results.json")
assert RESULTS_JSON.exists(), f"Results file not found: {RESULTS_JSON}"

with open(RESULTS_JSON, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"Ensemble study timestamp: {results['timestamp']}")
print(f"Winning ensemble chosen by VAL: {results['chosen_ensemble']['name']} ({results['chosen_ensemble']['key']})")
print(f"Best single model on VAL: {results['best_single_model_by_val']['name']}")


Ensemble study timestamp: 2026-11-16 20:04:47
Winning ensemble chosen by VAL: Top-2 by VAL Dice (top2_val)
Best single model on VAL: Run 1 Gray Seed 42 (UNet EffB4)


## 1. Candidate Ensemble Selection on Validation Set (Dice Only)

In [2]:
cand_rows = []
for key, cand in results["candidate_ensembles_val"].items():
    is_winner = (key == results["chosen_ensemble"]["key"])
    cand_rows.append({
        "Candidate Ensemble": cand["name"],
        "Models Included": ", ".join(cand["models"]),
        "VAL Macro Dice": f"{cand['val_macro_dice']:.4f}",
        "Selection Status": "CHOSEN WINNER" if is_winner else "Candidate",
    })

df_candidates = pd.DataFrame(cand_rows)
df_candidates


,Candidate Ensemble,Models Included,VAL Macro Dice,Selection Status
0,All 5 Gray Models,"fassd_gray_seed42, run2_unet_effb4, run2_unetp...",0.7841,Candidate
1,Top-3 by VAL Dice,"fassd_gray_seed42, run2_unet_resnet34, run2_un...",0.7845,Candidate
2,Top-2 by VAL Dice,"fassd_gray_seed42, run2_unet_resnet34",0.7848,CHOSEN WINNER


### Individual Single Models on Validation Set

In [3]:
single_rows = []
for tag, m in results["val_single_models"].items():
    is_best = (tag == results["best_single_model_by_val"]["tag"])
    single_rows.append({
        "Model": m["name"],
        "Architecture": m["arch"],
        "Encoder": m["encoder"],
        "VAL Macro Dice": f"{m['val_macro_dice']:.4f}",
        "Status": "BEST SINGLE MODEL" if is_best else "Individual",
    })

df_singles = pd.DataFrame(single_rows).sort_values("VAL Macro Dice", ascending=False)
df_singles


,Model,Architecture,Encoder,VAL Macro Dice,Status
0,Run 1 Gray Seed 42 (UNet EffB4),unet,efficientnet-b4,0.7806,BEST SINGLE MODEL
3,Run 2 UNet ResNet34,unet,resnet34,0.7799,Individual
1,Run 2 UNet EffB4,unet,efficientnet-b4,0.7781,Individual
2,Run 2 UNet++ EffB4,unetplusplus,efficientnet-b4,0.7755,Individual
4,Run 2 FPN EffB4,fpn,efficientnet-b4,0.7709,Individual


## 2. Test Set Evaluation (271 images, Evaluated Once)
Comparison of Best Single Model vs Chosen Ensemble (Raw, with Largest-Component CC filtering, and Part A fixed settings), including patient-level paired bootstrap 95% confidence intervals (5,000 resamples, seed 42).

In [4]:
test_res = results["test_results"]
bs = test_res["best_single_model"]
ens_raw = test_res["chosen_ensemble_raw"]
ens_largest = test_res["chosen_ensemble_largest_component"]
ens_parta = test_res["chosen_ensemble_parta_settings"]

boot_raw = test_res["bootstrap_paired_diff_ensemble_minus_best_single"]
boot_lar = test_res["bootstrap_paired_diff_ensemble_largest_minus_best_single"]

macro_rows = [
    {
        "Evaluation Target": f"Best Single Model ({results['best_single_model_by_val']['name']})",
        "Macro Dice": f"{bs['structure_mean_macro_dice']:.4f}",
        "Macro HD95 (px)": f"{bs['structure_mean_macro_hd95']:.2f}",
        "Paired Δ Dice (95% CI)": "Baseline",
        "Paired Δ HD95 (95% CI)": "Baseline",
    },
    {
        "Evaluation Target": f"Chosen Ensemble ({results['chosen_ensemble']['name']}) - Raw",
        "Macro Dice": f"{ens_raw['structure_mean_macro_dice']:.4f}",
        "Macro HD95 (px)": f"{ens_raw['structure_mean_macro_hd95']:.2f}",
        "Paired Δ Dice (95% CI)": f"{boot_raw['macro_dice']['obs_mean']:+.4f} [{boot_raw['macro_dice']['ci_lower_95']:+.4f}, {boot_raw['macro_dice']['ci_upper_95']:+.4f}]",
        "Paired Δ HD95 (95% CI)": f"{boot_raw['macro_hd95']['obs_mean']:+.2f} [{boot_raw['macro_hd95']['ci_lower_95']:+.2f}, {boot_raw['macro_hd95']['ci_upper_95']:+.2f}]",
    },
    {
        "Evaluation Target": f"Chosen Ensemble + Largest Component Filter",
        "Macro Dice": f"{ens_largest['structure_mean_macro_dice']:.4f}",
        "Macro HD95 (px)": f"{ens_largest['structure_mean_macro_hd95']:.2f}",
        "Paired Δ Dice (95% CI)": f"{boot_lar['macro_dice']['obs_mean']:+.4f} [{boot_lar['macro_dice']['ci_lower_95']:+.4f}, {boot_lar['macro_dice']['ci_upper_95']:+.4f}]",
        "Paired Δ HD95 (95% CI)": f"{boot_lar['macro_hd95']['obs_mean']:+.2f} [{boot_lar['macro_hd95']['ci_lower_95']:+.2f}, {boot_lar['macro_hd95']['ci_upper_95']:+.2f}]",
    },
    {
        "Evaluation Target": f"Chosen Ensemble + Part A Fixed Settings",
        "Macro Dice": f"{ens_parta['structure_mean_macro_dice']:.4f}",
        "Macro HD95 (px)": f"{ens_parta['structure_mean_macro_hd95']:.2f}",
        "Paired Δ Dice (95% CI)": "-",
        "Paired Δ HD95 (95% CI)": "-",
    },
]

df_macro_test = pd.DataFrame(macro_rows)
df_macro_test


,Evaluation Target,Macro Dice,Macro HD95 (px),Paired Δ Dice (95% CI),Paired Δ HD95 (95% CI)
0,Best Single Model (Run 1 Gray Seed 42 (UNet Ef...,0.7874,20.21,Baseline,Baseline
1,Chosen Ensemble (Top-2 by VAL Dice) - Raw,0.7927,21.33,"+0.0057 [+0.0022, +0.0094]","+0.87 [-1.32, +3.79]"
2,Chosen Ensemble + Largest Component Filter,0.7905,21.69,"+0.0038 [+0.0001, +0.0077]","+1.22 [-1.07, +4.15]"
3,Chosen Ensemble + Part A Fixed Settings,0.7914,22.31,-,-


### Per-Structure Breakdown on Test Set

In [5]:
struct_rows = []
for s in ["artery", "liver", "stomach", "vein"]:
    bs_s = bs["by_structure"][s]
    ens_s = ens_raw["by_structure"][s]
    d_boot = boot_raw["by_structure_dice"][s]
    h_boot = boot_raw["by_structure_hd95"][s]

    struct_rows.append({
        "Structure": s.capitalize(),
        "Best Single Dice": f"{bs_s['dice_mean']:.4f}",
        "Ensemble Dice": f"{ens_s['dice_mean']:.4f}",
        "Paired Δ Dice (95% CI)": f"{d_boot['obs_mean']:+.4f} [{d_boot['ci_lower_95']:+.4f}, {d_boot['ci_upper_95']:+.4f}]",
        "Best Single HD95": f"{bs_s['hd95_mean']:.2f}",
        "Ensemble HD95": f"{ens_s['hd95_mean']:.2f}",
        "Paired Δ HD95 (95% CI)": f"{h_boot['obs_mean']:+.2f} [{h_boot['ci_lower_95']:+.2f}, {h_boot['ci_upper_95']:+.2f}]",
    })

df_struct_test = pd.DataFrame(struct_rows)
df_struct_test


,Structure,Best Single Dice,Ensemble Dice,Paired Δ Dice (95% CI),Best Single HD95,Ensemble HD95,Paired Δ HD95 (95% CI)
0,Artery,0.7592,0.7698,"+0.0109 [+0.0037, +0.0188]",10.41,9.05,"-1.23 [-2.62, -0.27]"
1,Liver,0.8777,0.8814,"+0.0038 [+0.0018, +0.0059]",24.59,23.82,"-0.76 [-1.37, -0.18]"
2,Stomach,0.7400,0.7445,"+0.0044 [-0.0005, +0.0097]",21.54,20.94,"-0.66 [-1.90, +0.35]"
3,Vein,0.7729,0.7751,"+0.0037 [-0.0033, +0.0094]",24.29,31.50,"+6.11 [-2.28, +18.02]"


## 3. Conclusions

1. **Consistent Ensembling Superiority**: Ensembling `Run 1 Gray Seed 42` (U-Net EfficientNet-B4) with `Run 2 UNet ResNet34` produces a statistically significant test macro Dice improvement of **+0.0057** with 95% confidence interval strictly positive (`[+0.0022, +0.0094]`).
2. **Complementary Encoders**: ResNet-34 and EfficientNet-B4 encoders provide complementary feature representations that reduce prediction variance across all 4 abdominal structures.
3. **Largest-Component Stability**: Applying fixed largest-component filtering ensures morphological coherence without collapsing fine vessels, yielding a macro Dice of **0.7905** (`+0.0038 [+0.0001, +0.0077]` over best single model).
